In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import GroupKFold
from xgboost import XGBRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [2]:
df = pd.read_csv(
    "../data/processed/featured_uhi_v2.csv"
)

In [3]:
# 1. Spatial blocks: cut the city into a 10x10 grid
n_bins = 10
df["lat_bin"] = pd.cut(df["Latitude"], bins=n_bins, labels=False)
df["lon_bin"] = pd.cut(df["Longitude"], bins=n_bins, labels=False)
df["spatial_block"] = df["lat_bin"].astype(str) + "_" + df["lon_bin"].astype(str)

# 2. Physics-only features (no Lat/Long, no engineered duplicates)
FEATURES = ["NDVI", "NDBI", "Elevation", "Population"]
X = df[FEATURES]
y = df["LST"]
groups = df["spatial_block"]

# 3. Spatial cross-validation with out-of-fold predictions
gkf = GroupKFold(n_splits=5)
oof_pred = np.zeros(len(y))

for tr, te in gkf.split(X, y, groups):
    m = XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        random_state=42
    )
    m.fit(X.iloc[tr], y.iloc[tr])
    oof_pred[te] = m.predict(X.iloc[te])

mae  = mean_absolute_error(y, oof_pred)
rmse = mean_squared_error(y, oof_pred) ** 0.5
r2   = r2_score(y, oof_pred)

print("XGBoost V3 - Spatial CV (honest generalisation)")
print(f"MAE  : {mae:.3f}")
print(f"RMSE : {rmse:.3f}")
print(f"R2   : {r2:.3f}")

# 4. Train final model on ALL data, save for later notebooks
final_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)
final_model.fit(X, y)
joblib.dump(final_model, "../outputs/lst_model_xgb_v3.pkl")
print("\nSaved -> outputs/lst_model_xgb_v3.pkl")

XGBoost V3 - Spatial CV (honest generalisation)
MAE  : 1.777
RMSE : 2.259
R2   : 0.510

Saved -> outputs/lst_model_xgb_v3.pkl
